### Преобразование данных датасета Titanic

In [17]:
import pandas as pd
import numpy as np

data = pd.read_csv("./train.csv")

In [18]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [19]:
data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [20]:

data[['Pclass', 'Survived']].groupby(['Pclass'], as_index=False).mean()

,Pclass,Survived
0,1,0.629630
1,2,0.472826
2,3,0.242363


In [21]:
data[['SibSp', 'Survived']].groupby(['SibSp'], as_index=False).mean()

,SibSp,Survived
0,0,0.345395
1,1,0.535885
2,2,0.464286
3,3,0.250000
4,4,0.166667
5,5,0.000000
6,8,0.000000


In [22]:
data[['Parch', 'Survived']].groupby(['Parch'], as_index=False).mean()

,Parch,Survived
0,0,0.343658
1,1,0.550847
2,2,0.500000
3,3,0.600000
4,4,0.000000
5,5,0.200000
6,6,0.000000


In [23]:
data[['Fare', 'Survived']].groupby(['Fare'], as_index=False).mean()

,Fare,Survived
0,0.0000,0.066667
1,4.0125,0.000000
2,5.0000,0.000000
3,6.2375,0.000000
4,6.4375,0.000000
...,...,...
243,227.5250,0.750000
244,247.5208,0.500000
245,262.3750,1.000000
246,263.0000,0.500000


In [24]:
data[['Embarked', 'Survived']].groupby(['Embarked'], as_index=False).mean()

,Embarked,Survived
0,C,0.553571
1,Q,0.389610
2,S,0.336957


In [25]:
data[['Sex', 'Survived']].groupby(['Sex'], as_index=False).mean()

,Sex,Survived
0,female,0.742038
1,male,0.188908


In [26]:
data.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [27]:
data = data.drop(["PassengerId", "Name", "Age", "Ticket", "Fare", "Cabin"],axis=1)
data = pd.get_dummies(data=data, columns=['Sex', 'Embarked'], prefix=['sex', 'embarked'])

In [28]:
X = data.drop('Survived',axis=1)
y = data['Survived']

X_train, y_train, X_test, y_test = X[:int(len(X) * (7 / 10))],  \
                                    y[:int(len(X) * (7 / 10))], \
                                    X[int(len(X) * (7 / 10)):], \
                                    y[int(len(X) * (7 / 10)):]

---

### Decision Tree

In [29]:
from decision_tree import *
from utils import *

cf = {
    "max_depth" : 5,
    "min_split" : 2
}

dt = DecisionTree(cf, feature_names=X_train.columns.tolist(), mode='classification')
dt.fit(np.array(X_train), np.array(y_train))

print("Дерево решений:")
dt.print_feature_importance()

print("\nПороги для фичей:")
feature_thresholds = dt.get_feature_thresholds()
for feature, thresholds in feature_thresholds.items():
    print(f"{feature}: {thresholds}")

y_pred = dt.predict(np.array(X_test))

print("Accuracy: {}".format(accuracy(np.array(y_test), y_pred)))
print("Avg harmonical: {}".format(avg_harmonical(np.array(y_test), y_pred)))

Дерево решений:
Decision: sex_female <= 0.000
|-- True:
|  Decision: Pclass <= 1.000
|  |-- True:
|  |  Decision: Parch <= 1.000
|  |  |-- True:
|  |  |  Decision: SibSp <= 0.000
|  |  |  |-- True:
|  |  |  |  Decision: embarked_C <= 0.000
|  |  |  |  |-- True:
|  |  |  |  |  Leaf: class=0
|  |  |  |  |__ False:
|  |  |  |     Leaf: class=0
|  |  |  |__ False:
|  |  |     Decision: embarked_C <= 0.000
|  |  |     |-- True:
|  |  |     |  Leaf: class=0
|  |  |     |__ False:
|  |  |        Leaf: class=1
|  |  |__ False:
|  |     Decision: SibSp <= 1.000
|  |     |-- True:
|  |     |  Decision: Parch <= 2.000
|  |     |  |-- True:
|  |     |  |  Leaf: class=1
|  |     |  |__ False:
|  |     |     Leaf: class=0
|  |     |__ False:
|  |        Leaf: class=0
|  |__ False:
|     Decision: Parch <= 0.000
|     |-- True:
|     |  Decision: embarked_C <= 0.000
|     |  |-- True:
|     |  |  Decision: embarked_Q <= 0.000
|     |  |  |-- True:
|     |  |  |  Leaf: class=0
|     |  |  |__ False:
|

---

### Random Forest

In [30]:
from random_forest import *
from utils import *

cf = {
    "max_depth" : 5,
    "min_split" : 2
}

rf = RandomForestClassifier(50, num_features="sqrt",cf=cf)
rf.fit(X_train, y_train)

y_pred = rf.predict(np.array(X_test))

print("Accuracy: {}".format(accuracy(np.array(y_test), y_pred)))
print("Avg harmonical: {}".format(avg_harmonical(np.array(y_test), y_pred)))

Accuracy: 0.7827715355805244
Avg harmonical: 0.6703910614525139


---

### Linear Classification

In [31]:
from linear_classifier import *

lc = LinearClassifier()
lc.fit(np.array(X_train), np.array(y_train))

y_pred = lc.predict(np.array(X_test))

print("Accuracy: {}".format(accuracy(np.array(y_test), y_pred)))
print("Avg harmonical: {}".format(avg_harmonical(np.array(y_test), y_pred)))

Accuracy: 0.7902621722846442
Avg harmonical: 0.6779661016949152


In [32]:
from knn import *

knnc = KNNClassifier(4)
knnc.fit(np.array(X_train), np.array(y_train))

y_pred = knnc.predict(np.array(X_test))

print("Accuracy: {}".format(accuracy(np.array(y_test), y_pred)))
print("Avg harmonical: {}".format(avg_harmonical(np.array(y_test), y_pred)))

Accuracy: 0.8052434456928839
Avg harmonical: 0.6971428571428571
